# Clean-split regeneration (read this first)

The split was cleaned on 2026-09-21. `data/sft_pairs.jsonl` and
`data/dpo_pairs_zeroshot.jsonl` are now drawn through `splits.training()`, which
excludes every held-out equation and asserts it. **The checkpoints from the previous
session were trained on the contaminated draw and must be discarded**, so the `data`
stage is not optional.

```bash
cd src
python rerun_all.py data cpu                      # CPU, ~3 min
python rerun_all.py sft dpo eval --repeat=4       # GPU, ~60 min
```

`sft dpo eval`, not `eval` alone. Evaluation is greedy and therefore deterministic:
`eval --repeat=4` re-scores one checkpoint four times and prints four identical
tables. The driver now refuses that combination. What `--repeat` measures even when
used correctly is variance across *training runs* on a fixed 45-problem benchmark,
not sampling variance across independent test sets.

The builders print the contamination on every run. Before training, confirm you see:

```
split: 300 training / 45 held-out, 0 overlapping equations
held-out contamination: 0/45
```

Then the agent experiment, on the new checkpoint and not before it exists:

```bash
python rerun_all.py agent       # GPU, ~50 min
python rerun_all.py ablation    # GPU, ~45 min
```

The evaluation now writes `results/raw/eval/attempts_<utc>.csv` with every attempt,
its verdict and the verifier's reason, plus which problems the DPO stage gained and
lost. That is what makes the previous run's `incomplete` jump from 1 to 14
diagnosable rather than merely visible.


# agentic-verifier-selfrefine: Colab GPU run

Runs the parts of this repo that need a real LLM + GPU:
1. Rebuild the DPO preference pairs (already included as `data/dpo_pairs.jsonl`, generated from the synthetic flawed solver -- this cell is only needed if you want a version generated by the real base model's own errors instead).
2. LoRA-DPO fine-tune `Qwen/Qwen2.5-0.5B-Instruct` on the pairs.
3. Evaluate base vs. fine-tuned zero-shot solve rate on a held-out problem set.

Runtime: T4 GPU is sufficient for a 0.5B model.

**Status: this has already been run once for real** (a live browser session against Colab, code pasted in directly rather than cloned, since this repo was not pushed to GitHub at the time). Results are in `../results/colab_run/training_log.md` and `../PROVENANCE.md`. This notebook is for reproducing that run, and assumes the repo has since been pushed to GitHub at the URL in the next cell -- if it hasn't, use the "no GitHub" fallback cell instead of the `git clone` cell.

In [ ]:
# Option A: repo is on GitHub
!pip install -q peft trl transformers accelerate bitsandbytes datasets sympy
!git clone https://github.com/saqarkpr/agentic-verifier-selfrefine.git
%cd agentic-verifier-selfrefine/src

In [ ]:
# Option B: repo is NOT on GitHub (no clone target) -- upload the zip instead.
# Skip Option A above if you use this. Zip the repo locally
# (`zip -r agentic-verifier-selfrefine.zip agentic-verifier-selfrefine`) and
# upload it here.
# !pip install -q peft trl transformers accelerate bitsandbytes datasets sympy
# from google.colab import files
# uploaded = files.upload()  # pick agentic-verifier-selfrefine.zip
# !unzip -q agentic-verifier-selfrefine.zip
# %cd agentic-verifier-selfrefine/src

In [ ]:
# Optional: regenerate preference pairs using the real base model's own
# agentic trajectories instead of the synthetic flawed solver.
# import build_preference_data, hf_solver
# from tasks import generate_problems
# problems = generate_problems(n_per_family=40, seed=0)
# pairs = build_preference_data.build_pairs(problems, hf_solver.attempt, seed=0)
# (write pairs to ../data/dpo_pairs_real.jsonl if you use these instead)

In [ ]:
!python train_lora_dpo.py

In [ ]:
!python evaluate_finetuned.py

## Redesign v2: prompt-matched SFT + DPO (fair follow-up to the null result above)

The run above (cells 1-5) trained DPO only on the *revision* prompt template (equation + prior wrong attempt + verifier feedback) and evaluated on the *zero-shot* prompt -- a mismatch that is the leading explanation for its null result (0.022 base vs. 0.000 fine-tuned; see `../results/colab_run/training_log.md`). This section is the fair follow-up promised there: it does not re-run the same setup expecting a different answer, it trains and evaluates on the same prompt shape throughout, in two stages:

1. **SFT warm-start** (`train_lora_sft.py`) on `../data/sft_pairs.jsonl` -- correct worked solutions in the exact zero-shot prompt format, teaching the capability directly (the base model's zero-shot solve rate was only 2.2%, so DPO alone has little to sharpen).
2. **LoRA-DPO on top of the SFT model** (`train_lora_dpo_zeroshot.py`) on `../data/dpo_pairs_zeroshot.jsonl` -- preference pairs built in the same zero-shot format, not the revision template.
3. **Evaluation** (`evaluate_redesign_v2.py`) reports three numbers on the same held-out set (45 problems, seed=99; note this is NOT a disjoint split -- 6 of the 45 also appear in the SFT training file, see PROVENANCE.md bug #7): Base, SFT-only, and SFT+DPO -- so it's possible to see which stage helped, not just a single before/after pair.

This section makes **no promise about the outcome**. If the result is still null, or only partially positive, that is what gets reported -- the same honesty commitment as the rest of this repo (see `PROVENANCE.md`).

In [ ]:
!python build_sft_data.py
!python build_dpo_pairs_zeroshot.py

In [ ]:
!python train_lora_sft.py

In [ ]:
!python train_lora_dpo_zeroshot.py

In [ ]:
!python evaluate_redesign_v2.py

## The controlled test: does the feedback CONTENT help, or just having more tries?

The turn-budget experiment in cells above uses the synthetic solver, whose error rate is *defined* to fall each turn and which never reads the verifier's feedback. It therefore cannot show that feedback causes improvement (see `PROVENANCE.md`, "An overclaim corrected").

This cell runs the test that can. Each problem is solved twice under **identical sampling seeds**: once where turns 2 and 3 see the previous wrong answer plus the verifier's feedback, and once where they just see the original prompt again and re-sample. Turn 1 is identical in both by construction, which is a built-in check that the pairing is intact.

The control is the essential part. Three attempts beat one attempt regardless of feedback, simply because three samples are three chances. Only the gap between the two conditions at the same turn budget isolates what the feedback text buys.

Runtime: roughly 45 problems x 3 turns x 2 conditions x 3 seeds. A few minutes per seed on a T4.

**No prediction is being made about the outcome.** Note in advance that the SFT/SFT+DPO models were trained hard on one prompt shape, so a revision prompt is out of distribution for them, and a null result here would say as much about narrow fine-tuning as about verifier feedback.

In [ ]:
!python run_llm_agent_experiment.py

### Diagnosing that result

The raw table above says feedback did not help. Before believing it, run the diagnostic. It tests a hypothesis recorded in `run_llm_agent_experiment.py`'s docstring **before** the experiment ran: that a revision prompt is out of distribution for a model fine-tuned on one rigid prompt shape, and would break its output format.

It did. Feedback produced unparsable answers 38.5% of the time at turn 2 against the control's 6.2%. Unparsable scores as wrong, so correcting for it reverses the sign of the comparison. The verdict becomes *inconclusive*, not null.

CPU only, a few seconds.

In [ ]:
!python analyze_agent_experiment.py

### Re-running with the repaired prompt

`REVISION_PROMPT_STYLE` now defaults to `"minimal"`, which keeps the first and last lines of the revision prompt byte-identical to the training prompt instead of rewriting the instruction line.

This writes to `llm_agent_experiment_minimal.csv`, a **different file**, so the original run's evidence cannot be overwritten. Whatever this shows, including another null, gets reported next to the first result rather than replacing it.

In [ ]:
!python run_llm_agent_experiment.py
!python analyze_agent_experiment.py ../results/llm_agent_experiment_minimal.csv